# audit_e07
Phase 2 Stage 1 CPU-only E07 original-caption coverage audit (zero raw image IO, zero GPU/model weights, zero CONFIRM draw materialization). Run cells 2, 3, and 4 in order. Configure only cell 3. Use the pinned Python 3.11/3.12 CPU runtime (`requirements-stage2-cpu.txt`).


In [ ]:
import json, os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/Research-phase2')
CODE_REF = 'phase2'  # Set to the approved Phase 2 Stage 1 branch or commit SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage 1 settings (edit only this cell).
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
# Explicit accepted Phase 1 Stage 4 release pointer (hash-pinned).
STAGE4_RELEASE_REF = {
    'path': 'results/stage4/releases/20261119T064951Z-07e6f916a67f46fdb88c1812763534bc/_COMPLETE.json',
    'sha256': 'eb5cad38deecdd34e57f88f54a8a4fd8c47d7686fc3d159c240cde657d27c94c',
}
# Optional explicit resume reference ({'path': 'phase2/audits/e07/.../_COMPLETE.json', 'sha256': '...'}) or None.
RESUME_REF = None


In [ ]:
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 2 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Mounted PHASE1_ROOT not found: {PHASE1_ROOT}')
release_manifest = PHASE1_ROOT / STAGE4_RELEASE_REF['path']
if not release_manifest.is_file():
    raise RuntimeError(f'Missing mounted Stage 4 release manifest: {release_manifest}')
command = [
    sys.executable, '-u', '-m', 'src.phase2.e07_audit',
    '--phase1-root', str(PHASE1_ROOT),
    '--stage4-release', json.dumps(STAGE4_RELEASE_REF),
    '--storage-scope', 'persistent-drive',
]
if RESUME_REF is not None:
    resume_arg = json.dumps(RESUME_REF) if isinstance(RESUME_REF, dict) else str(RESUME_REF)
    command.extend(['--resume-ref', resume_arg])
process = subprocess.Popen(
    command, cwd=REPO, env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
blockers = []
for line in process.stdout:
    print(line, end='', flush=True)
    try:
        payload = json.loads(line)
        if isinstance(payload, dict) and payload.get('status') == 'blocked':
            blockers.extend(payload.get('blockers', []))
    except json.JSONDecodeError:
        pass
code = process.wait()
if code != 0:
    detail = '; '.join(str(b) for b in blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'E07 caption audit failed ({code}): {detail}')
